In [1]:
from config import Bundle
import duckdb
import os
import requests

In [2]:
def get_modrinth_official_categories(b1) -> set[str]:
    """
    description:
        this function serves to get the official modrinth values that belong in the categories column.
        
    param(s):
        b1 (Bundle): bundle object of configs and variables
    """
    url = f"{b1.modrinth_base_url}/tag/loader"
    response = requests.get(url, headers=b1.headers, timeout=30)
    response.raise_for_status()

    payload = response.json()

    modrinth_official_categories = set()

    for item in payload:
        if isinstance(item, str):
            modrinth_official_categories.add(item.lower())
        elif isinstance(item, dict) and "name" in item:
            modrinth_official_categories.add(str(item["name"]).lower())

    return modrinth_official_categories

In [3]:
def validate_platform_loader_map(b1, modrinth_official_loaders: set) -> bool:
    flattened_platform_loaders = {
        loader
        for loader_set in b1.platform_loader_map.values()
        for loader in loader_set
    }

    official_loader_values = {
        str(loader).strip().lower()
        for loader in modrinth_official_loaders
    }

    platform_loaders_normalized = {
        str(loader).strip().lower()
        for loader in flattened_platform_loaders
    }

    missing_from_official = sorted(
        platform_loaders_normalized - official_loader_values
    )

    if missing_from_official:
        print("Found platform_loader_map values missing from official Modrinth loaders:")
        for value in missing_from_official:
            print(f"\t  - {value}")
        return False
    else:
        print("b1.platform_loader_map matches official Modrinth loaders.")
        return True

In [4]:
def to_sql_string_list(values: set[str]) -> str:
    escaped_values = []
    
    for value in sorted(values):
        clean_value = str(value).strip().lower().replace("'", "''")
        escaped_values.append(f"'{clean_value}'")
    
    return ", ".join(escaped_values)

In [5]:
def build_platform_loaders_case_sql(b1) -> str:
    case_clauses = []

    for project_type, loader_values in b1.platform_loader_map.items():
        loader_sql = to_sql_string_list(loader_values)

        case_clause = f"""
        WHEN LOWER(b.project_type) = '{project_type.lower()}' THEN COALESCE((
            SELECT LIST(DISTINCT LOWER(category_value) ORDER BY LOWER(category_value))
            FROM UNNEST(b.categories) AS t(category_value)
            WHERE LOWER(category_value) IN ({loader_sql})
            ), []::VARCHAR[])
        """
        case_clauses.append(case_clause)

    return f"""
            CASE
                {''.join(case_clauses)}
                ELSE []::VARCHAR[]
            END
    """

In [6]:
def create_snapshot_project_listings(b1) -> None:
    platform_loaders_sql = build_platform_loaders_case_sql(b1)

    source_count_sql = f"""
        SELECT COUNT(*) AS row_count
        FROM silver_db.{b1.base_api_project_listings_table_name}
    """

    create_snapshot_sql = f"""
        CREATE OR REPLACE TABLE {b1.snapshot_api_project_listings_table_name} AS
        WITH base AS (
            SELECT *
            FROM silver_db.{b1.base_api_project_listings_table_name}
        ),
        with_platform_loaders AS (
            SELECT
                b.run_id,
                b.project_type,
                b.project_id,
                b.slug,
                b.display_title,
                b.author,
                b.description,
                b.categories,
                {platform_loaders_sql} AS platform_loaders,
                b.versions,
                b.latest_version,
                b.download_count,
                b.follows,
                b.client_side,
                b.server_side,
                b.license,
                b.date_created,
                b.date_modified,
                b.date_retrieved_at
            FROM base b
        ),
        with_game_categories AS (
            SELECT
                wpl.run_id,
                wpl.project_type,
                wpl.project_id,
                wpl.slug,
                wpl.display_title,
                wpl.author,
                wpl.description,
                wpl.categories,
                wpl.platform_loaders,
                COALESCE((
                    SELECT LIST(category_value ORDER BY category_value)
                    FROM UNNEST(wpl.categories) AS t(category_value)
                    WHERE LOWER(category_value) NOT IN (
                        SELECT LOWER(loader_value)
                        FROM UNNEST(wpl.platform_loaders) AS p(loader_value)
                    )
                ), []::VARCHAR[]) AS gameplay_categories,
                wpl.versions,
                wpl.latest_version,
                wpl.download_count,
                wpl.follows,
                wpl.client_side,
                wpl.server_side,
                wpl.license,
                wpl.date_created,
                wpl.date_modified,
                wpl.date_retrieved_at
            FROM with_platform_loaders wpl
        )
        SELECT *
        FROM with_game_categories
    """

    target_count_sql = f"""
        SELECT COUNT(*) AS row_count
        FROM {b1.snapshot_api_project_listings_table_name}
    """

    with duckdb.connect(b1.gold_db_path) as gold_con:
        gold_con.execute(f"ATTACH '{b1.silver_db_path}' AS silver_db")

        source_row_count = gold_con.execute(source_count_sql).fetchone()[0]
        print(f"Source row count from silver: {source_row_count:,}")

        gold_con.execute(create_snapshot_sql)

        target_row_count = gold_con.execute(target_count_sql).fetchone()[0]
        print(f"Target row count in gold snapshot: {target_row_count:,}")

        if source_row_count != target_row_count:
            raise ValueError(
                f"Row count mismatch detected. "
                f"Source rows: {source_row_count:,}, "
                f"Target rows: {target_row_count:,}. "
                f"Transformation may have dropped or duplicated rows."
            )

        print("Row count validation passed. No rows were dropped during transformation.")

In [8]:
with duckdb.connect(b1.silver_db_path) as silver_con:
    df = silver_con.execute(f"SELECT * FROM {b1.base_api_project_listings_table_name}").fetch_df()

In [9]:
df

,run_id,project_type,project_id,c_pull_timestamp_utc,all_project_types,slug,author,author_id,organization,organization_id,...,date_modified,latest_version,license,client_side,server_side,environment,disclosure_types,gallery,featured_gallery,color
0,21f1554d-0f8b-423c-b708-97f80453e17f,shader,HVnmMxH1,2026-08-21 19:09:04.103089-07:00,[shader],complementary-reimagined,EminGT,O8iu3LjK,NaN,NaN,...,2026-05-21T14:50:00.489808+00:00,yCCduG44,LicenseRef-Custom,required,unsupported,[unknown],[],[https://cdn.modrinth.com/data/HVnmMxH1/images...,https://cdn.modrinth.com/data/HVnmMxH1/images/...,14197949
1,21f1554d-0f8b-423c-b708-97f80453e17f,shader,R6NEzAwj,2026-08-21 19:09:04.103089-07:00,[shader],complementary-unbound,EminGT,O8iu3LjK,NaN,NaN,...,2026-05-21T14:50:31.396444+00:00,VMHXIk50,LicenseRef-Custom,required,unsupported,[unknown],[],[https://cdn.modrinth.com/data/R6NEzAwj/images...,https://cdn.modrinth.com/data/R6NEzAwj/images/...,2630468
2,21f1554d-0f8b-423c-b708-97f80453e17f,shader,Q1vvjJYV,2026-08-21 19:09:04.103089-07:00,[shader],bsl-shaders,CaptTatsu,a239L3bu,NaN,NaN,...,2026-04-20T13:19:23.539323+00:00,hIibTfxn,LicenseRef-All-Rights-Reserved,required,unsupported,[unknown],[],[https://cdn.modrinth.com/data/Q1vvjJYV/images...,https://cdn.modrinth.com/data/Q1vvjJYV/images/...,6071540
3,21f1554d-0f8b-423c-b708-97f80453e17f,shader,lLqFfGNs,2026-08-21 19:09:04.103089-07:00,[shader],photon-shader,sixthsurge,hdz0KMgX,NaN,NaN,...,2026-04-14T20:47:00.122415+00:00,gUv7fBPN,LicenseRef-,required,unsupported,[],[],[https://cdn.modrinth.com/data/lLqFfGNs/images...,https://cdn.modrinth.com/data/lLqFfGNs/images/...,10973017
4,21f1554d-0f8b-423c-b708-97f80453e17f,shader,EpQFjzrQ,2026-08-21 19:09:04.103089-07:00,[shader],solas-shader,Septonious,dilHPVn4,NaN,NaN,...,2026-07-01T14:38:52.841402+00:00,WcoEHPPx,LicenseRef-All-Rights-Reserved,required,unsupported,[unknown],[],[https://cdn.modrinth.com/data/EpQFjzrQ/images...,https://cdn.modrinth.com/data/EpQFjzrQ/images/...,15969902
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
157752,21f1554d-0f8b-423c-b708-97f80453e17f,mod,IatB32KG,2026-08-21 19:07:50.310225-07:00,[mod],alternative-lamps-bta,camoweed,9GNeNJ72,NaN,NaN,...,2026-08-15T05:39:38.521598+00:00,1l9PiAYM,CC0-1.0,required,required,[client_and_server],[],[https://cdn.modrinth.com/data/IatB32KG/images...,https://cdn.modrinth.com/data/IatB32KG/images/...,14982220
157753,21f1554d-0f8b-423c-b708-97f80453e17f,mod,MCbQXb3t,2026-08-21 19:07:50.310225-07:00,[mod],areax,anmao,xDLAesqZ,NaN,NaN,...,2026-08-15T02:19:57.915762+00:00,no9KCHMl,CC-BY-SA-4.0,required,required,[client_and_server],[],[],NaN,263172
157754,21f1554d-0f8b-423c-b708-97f80453e17f,mod,hEazeeIK,2026-08-21 19:07:50.310225-07:00,[mod],limitless_command,NarcissoOrigin,VSSPXuXr,NaN,NaN,...,2026-08-06T04:36:55.387888+00:00,damDvz27,MIT,required,required,[client_and_server],[],[],NaN,5520688
157755,21f1554d-0f8b-423c-b708-97f80453e17f,mod,a73QkAye,2026-08-21 19:07:50.310225-07:00,[mod],friendly_pets,NarcissoOrigin,VSSPXuXr,NaN,NaN,...,2026-08-06T02:37:04.966828+00:00,AvMihunL,MIT,required,required,[client_and_server],[],[],NaN,13149848


In [10]:
with duckdb.connect(b1.gold_db_path) as gold_con:
    df = gold_con.execute(f"SELECT * FROM {b1.snapshot_api_project_listings_table_name}").fetch_df()

In [11]:
df

,run_id,project_type,project_id,slug,display_title,author,description,categories,platform_loaders,gameplay_categories,versions,latest_version,download_count,follows,client_side,server_side,license,date_created,date_modified,date_retrieved_at
0,34f1e778-3737-4738-8588-139493582b09,mod,quBwUCpJ,pmweather-misc-additions,PMWeather: MISC Additions,Interchange_The_Protogen,This is an addon to protomanly's weather mod t...,"[adventure, decoration, equipment, mobs, neofo...",[neoforge],"[adventure, decoration, equipment, mobs, world...",[1.21.1],76LDO59i,827,1,unsupported,required,LicenseRef-All-Rights-Reserved,2026-02-17 05:27:14.861388,2026-02-26 05:11:12.388786,2026-03-26 00:23:47.356473
1,34f1e778-3737-4738-8588-139493582b09,mod,rXz0zuOD,wakfu,Wakfu,1_1oxiytb1705,A mod on the theme of the anime/ game Wakfu,"[adventure, equipment, forge]",[forge],"[adventure, equipment]",[1.20.1],Va7J8s8j,827,3,required,required,LicenseRef-All-Rights-Reserved,2024-05-30 00:01:08.078461,2026-01-28 11:56:32.776445,2026-03-26 00:23:47.356475
2,34f1e778-3737-4738-8588-139493582b09,mod,arl5PIDq,gardons-vehicles-pack,Gardon's Vehicles Pack,Gardon,New Vehicles to Minecraft!,"[decoration, equipment, forge, game-mechanics,...","[forge, neoforge]","[decoration, equipment, game-mechanics, librar...","[1.20.1, 1.21.8]",qcDkj2I9,827,2,required,required,LicenseRef-All-Rights-Reserved,2025-05-05 21:21:30.812627,2025-12-24 13:27:10.115981,2026-03-26 00:23:47.356476
3,34f1e778-3737-4738-8588-139493582b09,mod,OKTaOxQM,sixsevenstack,SixSeven,Tokimi,Who decided stacks should stop at 64? SixSeven...,"[cursed, fabric, game-mechanics, management]",[fabric],"[cursed, game-mechanics, management]","[1.20, 1.20.1, 1.20.2, 1.20.3, 1.20.4, 1.21, 1...",Q3Oh2vAw,827,4,required,required,LicenseRef-All-Rights-Reserved,2025-12-06 03:26:11.501770,2025-12-04 01:43:47.045809,2026-03-26 00:23:47.356479
4,34f1e778-3737-4738-8588-139493582b09,mod,vIhn4QvG,nblb,No Block Left Behind,DeltaV,Adds block variants for the blocks that don't ...,"[decoration, fabric]",[fabric],[decoration],"[1.21.5, 1.21.6, 1.21.7, 1.21.9, 1.21.10]",8RwE5gme,827,8,required,required,CC0-1.0,2025-06-27 01:44:16.401072,2025-10-14 03:07:17.738996,2026-03-26 00:23:47.356480
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
121137,bb9c75c2-56c0-42b8-9020-61f0a68f3e23,plugin,gMGrRLqi,blacks-random-loot,Black's Random Loot,_blackdev_,Generates Random Loot in a Chest named with th...,"[bukkit, equipment, minigame, paper, spigot, s...","[bukkit, paper, spigot]","[equipment, minigame, storage, utility]","[1.8.8, 1.12.2, 1.16.5, 1.20]",Ja7BA7Ra,466,1,unsupported,required,LicenseRef-All-Rights-Reserved,2024-09-14 22:50:52.426257,2024-09-14 16:01:22.721983,2026-03-26 00:36:31.887980
121138,bb9c75c2-56c0-42b8-9020-61f0a68f3e23,plugin,eT7sT4Kc,xpdispenser,XPDispenser,Programie,A Bukkit plugin to throw your XP at yourself (...,"[bukkit, game-mechanics, paper, spigot, storage]","[bukkit, paper, spigot]","[game-mechanics, storage]","[1.14, 1.14.1, 1.14.2, 1.14.3, 1.14.4, 1.15, 1...",2n910QZr,466,1,unsupported,required,MIT,2023-05-25 21:08:37.103659,2024-01-30 21:56:33.169389,2026-03-26 00:36:31.887982
121139,bb9c75c2-56c0-42b8-9020-61f0a68f3e23,plugin,BxpRVNai,foxenchantcreator,FoxEnchantCreator,LisooYT,Custom Enchant Creator that allows you to easi...,"[bukkit, equipment, game-mechanics, library, m...","[bukkit, paper, purpur, spigot]","[equipment, game-mechanics, library, magic, ma...","[1.21, 1.21.1, 1.21.2, 1.21.3, 1.21.4, 1.21.5,...",1olwTrjH,465,7,unsupported,required,LicenseRef-All-Rights-Reserved,2025-06-08 06:40:00.691043,2026-01-01 11:18:15.707714,2026-03-26 00:36:31.887984
121140,bb9c75c2-56c0-42b8-9020-61f0a68f3e23,plugin,DbsSQeTR,random-events,Random Events,nesteds,Random events occur periodically to keep the p...,"[adventure, cursed, game-mechanics, magic, min...","[paper, purpur, spigot]","[adventure, cursed, game-mechanics, magic, min...","[1.20, 1.20.1, 1.20.2, 1.20.3, 1.20.4, 1.20

In [7]:
#initialize object
b1 = Bundle()

#create gold layer with databases and tables initialized
b1.build_layer_directory(b1.gold_env_folder_path)
b1.init_db('gold')

#get modrinth offical category items and validate before transformation
    #[forge, fabric, spigot, bukkit, etc]
modrinth_official_categories = get_modrinth_official_categories(b1)
if not validate_platform_loader_map(b1, modrinth_official_categories):
    raise ValueError(
        "Halting notebook. Please investigate b1.platform_loader_map values before proceeding with transformations."
    )

#transform project_listings table: categories -> platform_loaders
create_snapshot_project_listings(b1)

b1.platform_loader_map matches official Modrinth loaders.
Source row count from silver: 157,757


BinderException: Binder Error: Values list "b" does not have a column named "display_title"

LINE 13:                 b.display_title,
                         ^